
###### Project: 

- End-to-End Agentic AI Customer Churn Platform

###### Phase:
- GenAI - Retrieval Augmented Generation

###### Previous Notebook:
- 11_Vector_Search

###### Current Notebook:
- 12_RAG_Chatbot

###### Next Notebook:
- 13_Agent_Tools


###### 12_RAG_Chatbot

###### Purpose

This notebook retrieves semantically relevant customer notes using Databricks Vector Search and generates grounded responses with a Large Language Model (LLM) through Retrieval-Augmented Generation (RAG).


###### Business Problem

Customer support teams need quick and accurate answers based on historical customer interactions. Traditional keyword search often misses semantically similar cases, resulting in incomplete or less relevant responses. This notebook addresses that challenge by implementing Retrieval-Augmented Generation (RAG), which combines semantic search over customer notes with a Large Language Model (LLM) to generate grounded, context-aware responses.


###### Technologies Used

- Databricks

- Delta Lake

- Unity Catalog

- Databricks Vector Search

- Databricks Embedding Foundation Model (databricks-gte-large-en)

- LLM Model (databricks-meta-llama-3-1-8b-instruct)

###### Input

- Existing Vector Search Endpoint

- Existing Vector Search Index

- Delta table with precomputed embeddings

- User question

- Embedding Model

- LLM Model

- Prompt

######  Output

- LLM generated grounded response

######  Architecture

```text

User Question
       ↓
Embedding Model
      ↓
Question Embedding
       ↓
Existing Vector Search Index
      ↓
Top-k Relevant customer notes
       ↓
Build Context
       ↓
Prompt 
       ↓
Large Language Model (LLM)
       ↓
Grounded Answer
     
```


###### Section 0 : Installation

In [0]:
#Install Vector Search client
%pip install databricks-vectorsearch
dbutils.library.restartPython()

###### Section 1 : Call project config notebook

In [0]:
%run ./00_project_config

###### Section 2 : Imports libraries

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()
vsc = VectorSearchClient()
vsc.list_endpoints()

###### Section 3 : Connect to Existing Vector Search Index

In [0]:
# Load the existing Vector Search index created in Notebook 11
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=VECTOR_INDEX_NAME
)

print(index.describe())

###### Section 4 : Create a retrieval function

In [0]:
def retrieve_context(question, num_results=3):

    if not question.strip():
        raise ValueError(
            "Question cannot be empty."
        )

    # Generate query embedding
    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    if (
        not response.data
        or response.data[0].embedding is None
    ):
        raise ValueError(
            "Embedding model did not return an embedding."
        )

    question_embedding = [
        float(x)
        for x in response.data[0].embedding
    ]

    # Semantic search
    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    rows = (
        results
        .get("result", {})
        .get("data_array", [])
    )

    if not rows:
        return "", []

    context_lines = []

    for customer_id, note, score in rows:
        context_lines.append(
            f"Customer {int(customer_id)}: {note}"
        )

    return "\n".join(context_lines), rows

###### Section 5 :  Build the prompt

In [0]:
def build_prompt(question, context):

    prompt = f"""
You are a telecom customer-support assistant.

Answer the question using ONLY the retrieved customer notes provided below.
Do not use outside knowledge or make assumptions.

If the retrieved customer notes do not contain enough information to answer the question, respond exactly:

"I don't have enough information from the retrieved customer notes."

Keep the answer concise and factual.

Retrieved Customer Notes:
{context}

Question:
{question}

Answer:
"""

    return prompt

###### Section 6 :  Call the LLM

In [0]:
def generate_answer(prompt):
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=256,
        temperature=0
    )

    return response.choices[0].message.content

###### Section 7 :  End-to-end RAG function

In [0]:
def rag_answer(question):
    # retrieve and  build context
    context, retrieved_rows = retrieve_context(
        question=question,
        num_results=3
    )
    
    #build prompt
    prompt = build_prompt(
        question=question,
        context=context
    )
    
    #call LLM
    answer = generate_answer(prompt)

    return answer, context, retrieved_rows

###### Section 8 :  Test RAG

In [0]:
questions = [
    "Why are customers likely to cancel service?",
    "Why are customers cancelling?",
    "Customer wants to upgrade plan.",
    "Billing complaints?"
]

for i, question in enumerate(questions, start=1):

    answer, context, retrieved_rows = rag_answer(question)

    print("=" * 80)
    print(f"Question {i}")
    print("=" * 80)

    print("QUESTION:")
    print(question)

    print("\nRETRIEVED CONTEXT:")
    print(context)

    print("\nANSWER:")
    print(answer)

    print()

###### Notebook Summary

- Get Configurations for Vector Search endpoint name,  Vector index name,  Embedding model name and LLM endpoint name.

- Connect to the existing Vector Search Index.

- Create Retrieval Function to Convert question to embedding, Query Vector Search and Return top-k customer notes.

- Build Context Function to Convert retrieved rows into context text.

- Build Prompt Function to System instruction, Retrieved context and User question.

- Create Function to Call LLM

- Create RAG Function to retrieve,  build context,  build prompt and call LLM

- Test RAG

###### Key Learnings

- RAG combines retrieval and generation.

- Vector Search retrieves relevant context before calling the LLM.

- Prompt quality affects RAG answer quality.

- Grounded answers should use retrieved context instead of relying only on the LLM’s general knowledge.

- RAG separates knowledge retrieval from language generation.

###### Notebook Conclusion

- In this notebook, we built a Retrieval-Augmented Generation workflow that retrieves relevant customer note context using Vector Search and passes the retrieved customer context to an LLM to produce grounded, project-specific responses.

- This enables grounded answer generation using project-specific customer notes data instead of relying only on the LLM's general knowledge.

- This will be used in the next notebook to extend the RAG workflow into an Agentic AI application.

###### Next Notebook

13_Rule_based_Agent

The purpose of this notebook is to build reusable AI tools (Vector Search, Customer Lookup, Churn Prediction, Billing Information, etc.) that an AI Agent can dynamically select and invoke.